# Homework 2: Data Cleaning

This notebook cleans and combines World Bank data about access to electricity and GDP per capita. The final dataset will include country-level data from 2000 to 2023.

In [1]:
import pandas as pd
import json

from urllib.request import urlopen
from pathlib import Path

In [2]:
raw_data_path = Path("../Data/raw")

list(raw_data_path.glob("*.csv"))

[WindowsPath('../Data/raw/WB_WDI_EG_ELC_ACCS_ZS_WIDEF.csv'),
 WindowsPath('../Data/raw/WB_WDI_NY_GDP_PCAP_KD_WIDEF.csv')]

In [3]:
electricity_df = pd.read_csv(
    raw_data_path / "WB_WDI_EG_ELC_ACCS_ZS_WIDEF.csv"
)

gdp_df = pd.read_csv(
    raw_data_path / "WB_WDI_NY_GDP_PCAP_KD_WIDEF.csv"
)

print("Electricity dataset shape:", electricity_df.shape)
print("GDP dataset shape:", gdp_df.shape)

Electricity dataset shape: (263, 73)
GDP dataset shape: (260, 105)


In [4]:
preview_columns = [
    "REF_AREA",
    "REF_AREA_LABEL",
    "2000",
    "2010",
    "2020",
    "2023"
]

print("Electricity data:")
display(electricity_df[preview_columns].head())

print("GDP per capita data:")
display(gdp_df[preview_columns].head())

Electricity data:


,REF_AREA,REF_AREA_LABEL,2000,2010,2020,2023
0,ITA,Italy,100.0,100.0,100.0,100.0
1,BTN,Bhutan,31.2,73.3,100.0,100.0
2,SDN,Sudan,23.0,36.0,59.7,66.0
3,EGY,"Egypt, Arab Rep.",97.7,99.4,100.0,100.0
4,MUS,Mauritius,99.0,99.4,99.5,100.0


GDP per capita data:


,REF_AREA,REF_AREA_LABEL,2000,2010,2020,2023
0,IBT,IDA & IBRD total,2340.828752,3721.631190,4975.192051,5634.164827
1,CHE,Switzerland,74933.277008,83069.427637,86293.918344,92947.942444
2,ARG,Argentina,10631.650364,13387.155375,11393.050596,12993.092887
3,GRC,Greece,18002.906461,21547.724178,17486.951333,21050.864811
4,CEB,Central Europe and the Baltics,7450.883715,10998.682372,14621.174025,16496.672878


##  Keep the Needed Data

We removed extra columns and kept the country code, country name, and data from 2000 to 2023.

In [5]:
years = [str(year) for year in range(2000, 2024)]

columns_to_keep = [
    "REF_AREA",
    "REF_AREA_LABEL"
] + years

electricity_clean = electricity_df[columns_to_keep].copy()
gdp_clean = gdp_df[columns_to_keep].copy()

print("Electricity data after selecting columns:", electricity_clean.shape)
print("GDP data after selecting columns:", gdp_clean.shape)

Electricity data after selecting columns: (263, 26)
GDP data after selecting columns: (260, 26)


##  Reshape the Data

We reorganized the data so each row shows one country for one year. This makes the two datasets easier to combine and study.


In [6]:
electricity_long = electricity_clean.melt(
    id_vars=["REF_AREA", "REF_AREA_LABEL"],
    var_name="year",
    value_name="electricity_access"
)

gdp_long = gdp_clean.melt(
    id_vars=["REF_AREA", "REF_AREA_LABEL"],
    var_name="year",
    value_name="gdp_per_capita"
)

print("Electricity long shape:", electricity_long.shape)
print("GDP long shape:", gdp_long.shape)

display(electricity_long.head())
display(gdp_long.head())

Electricity long shape: (6312, 4)
GDP long shape: (6240, 4)


,REF_AREA,REF_AREA_LABEL,year,electricity_access
0,ITA,Italy,2000,100.0
1,BTN,Bhutan,2000,31.2
2,SDN,Sudan,2000,23.0
3,EGY,"Egypt, Arab Rep.",2000,97.7
4,MUS,Mauritius,2000,99.0


,REF_AREA,REF_AREA_LABEL,year,gdp_per_capita
0,IBT,IDA & IBRD total,2000,2340.828752
1,CHE,Switzerland,2000,74933.277008
2,ARG,Argentina,2000,10631.650364
3,GRC,Greece,2000,18002.906461
4,CEB,Central Europe and the Baltics,2000,7450.883715


## Standardize Columns and Data Types

We will give the columns clearer names and make sure the year and measurement columns use numeric data types.

In [7]:
electricity_long = electricity_long.rename(
    columns={
        "REF_AREA": "country_code",
        "REF_AREA_LABEL": "country_name"
    }
)

gdp_long = gdp_long.rename(
    columns={
        "REF_AREA": "country_code",
        "REF_AREA_LABEL": "country_name"
    }
)

electricity_long["year"] = pd.to_numeric(
    electricity_long["year"], errors="coerce"
)

electricity_long["electricity_access"] = pd.to_numeric(
    electricity_long["electricity_access"], errors="coerce"
)

gdp_long["year"] = pd.to_numeric(
    gdp_long["year"], errors="coerce"
)

gdp_long["gdp_per_capita"] = pd.to_numeric(
    gdp_long["gdp_per_capita"], errors="coerce"
)

print(electricity_long.dtypes)
print()
print(gdp_long.dtypes)

country_code              str
country_name              str
year                    int64
electricity_access    float64
dtype: object

country_code          str
country_name          str
year                int64
gdp_per_capita    float64
dtype: object


## Combine the Datasets

The datasets will be joined using the country code and year. An inner join will keep records that appear in both datasets.

In [8]:
combined_df = electricity_long.merge(
    gdp_long[["country_code", "year", "gdp_per_capita"]],
    on=["country_code", "year"],
    how="inner",
    validate="one_to_one"
)

In [9]:
combined_df.shape

(6216, 5)

## Check for missing values

Missing values are places where information was not recorded. We will count the missing values in each column before deciding how to handle them.

In [31]:
combined_df.isna().sum()

country_code            0
country_name            0
year                    0
electricity_access     40
gdp_per_capita        149
dtype: int64

In [32]:
missing_rows = combined_df[
    ["electricity_access", "gdp_per_capita"]
].isna().any(axis=1).sum()

missing_rows

np.int64(174)

In [33]:
missing_percent = (missing_rows / len(combined_df)) * 100

missing_percent

np.float64(2.799227799227799)

##  Remove Incomplete Rows

About 2.8% of the combined rows are missing an electricity-access value, a GDP-per-capita value, or both. These rows will be removed because both measurements are needed for the comparison.

In [34]:
cleaned_df = combined_df.dropna(
    subset=["electricity_access", "gdp_per_capita"]
).copy()

cleaned_df.shape

(6042, 5)

##  Identify Individual Countries

The data included countries and regional groups. We used World Bank country information to keep only individual countries and territories.

In [35]:
country_url = "https://" + "api.worldbank.org/v2/country?format=json&per_page=400"

In [36]:
with urlopen(country_url) as response:
    country_data = json.load(response)

In [37]:
country_records = country_data[1]
len(country_records)

295

In [38]:
country_codes = []

for record in country_records:
    if record["region"]["value"] != "Aggregates":
        country_codes.append(record["id"])

In [39]:
len(country_codes)

217

In [40]:
countries_only_df = cleaned_df[
    cleaned_df["country_code"].isin(country_codes)
].copy()

countries_only_df.shape

(4914, 5)

In [41]:
duplicate_rows = countries_only_df.duplicated(
    subset=["country_code", "year"]
).sum()

duplicate_rows

np.int64(0)

In [42]:
print(
    "Electricity access range:",
    countries_only_df["electricity_access"].min(),
    "to",
    countries_only_df["electricity_access"].max()
)

print(
    "GDP per capita range:",
    countries_only_df["gdp_per_capita"].min(),
    "to",
    countries_only_df["gdp_per_capita"].max()
)

Electricity access range: 1.3 to 100.0
GDP per capita range: 233.0323930863032 to 225884.0771991251


In [43]:
electricity_is_valid = countries_only_df[
    "electricity_access"
].between(0, 100).all()

gdp_is_valid = (
    countries_only_df["gdp_per_capita"] >= 0
).all()

print("All electricity values are valid:", electricity_is_valid)
print("All GDP values are valid:", gdp_is_valid)

All electricity values are valid: True
All GDP values are valid: True


In [44]:
display(
    countries_only_df.nsmallest(
        3, "electricity_access"
    )[["country_name", "year", "electricity_access"]]
)

display(
    countries_only_df.nlargest(
        3, "gdp_per_capita"
    )[["country_name", "year", "gdp_per_capita"]]
)

,country_name,year,electricity_access
322,Lesotho,2001,1.3
717,Guinea-Bissau,2002,1.3
2170,Liberia,2008,1.3


,country_name,year,gdp_per_capita
6052,Monaco,2023,225884.077199
5793,Monaco,2022,214359.505135
5534,Monaco,2021,194674.686164


##  Validate the Measurement Ranges

We checked the electricity and GDP values. All values were within reasonable ranges, and no impossible values were found.

In [45]:
final_df = countries_only_df.sort_values(
    by=["country_name","year"]
).reset_index(drop=True)

display(final_df.head(10))


,country_code,country_name,year,electricity_access,gdp_per_capita
0,AFG,Afghanistan,2000,4.4,308.318270
1,AFG,Afghanistan,2001,9.3,277.118051
2,AFG,Afghanistan,2002,14.1,338.139974
3,AFG,Afghanistan,2003,19.0,346.071627
4,AFG,Afghanistan,2004,23.8,338.637274
5,AFG,Afghanistan,2005,28.7,363.640141
6,AFG,Afghanistan,2006,33.5,367.758312
7,AFG,Afghanistan,2007,38.4,410.757729
8,AFG,Afghanistan,2008,42.4,417.647283
9,AFG,Afghanistan,2009,48.3,488.830652


In [46]:
print("Number of rows:", len(final_df))
print("Number of columns:", final_df.shape[1])
print("Number of countries:", final_df["country_code"].nunique())
print("First year:", final_df["year"].min())
print("Last year:", final_df["year"].max())
print("Missing values:",final_df.isna().sum().sum())
print(
    "Duplicate country-year rows:",
    final_df.duplicated(
        subset=["country_code", "year"]
    ).sum()
)


Number of rows: 4914
Number of columns: 5
Number of countries: 211
First year: 2000
Last year: 2023
Missing values: 0
Duplicate country-year rows: 0


##  Save the Cleaned Dataset

The final data matches electricity and GDP information by country and year from 2000 to 2023. Missing values, duplicates, and regional groups were removed.

In [47]:
cleaned_data_path = Path("../Data/cleaned")

output_file = cleaned_data_path / "electricity_gdp_cleaned.csv"

final_df.to_csv(output_file, index=False)

print("Cleaned dataset saved to:", output_file)

Cleaned dataset saved to: ..\Data\cleaned\electricity_gdp_cleaned.csv


In [48]:
saved_df = pd.read_csv(output_file)

print("Saved file shape:", saved_df.shape)
print("Missing values:", saved_df.isna().sum().sum())
print(
    "Duplicate country-year rows:",
    saved_df.duplicated(
        subset=["country_code", "year"]
    ).sum()
)

display(saved_df.head())

Saved file shape: (4914, 5)
Missing values: 0
Duplicate country-year rows: 0


,country_code,country_name,year,electricity_access,gdp_per_capita
0,AFG,Afghanistan,2000,4.4,308.318270
1,AFG,Afghanistan,2001,9.3,277.118051
2,AFG,Afghanistan,2002,14.1,338.139974
3,AFG,Afghanistan,2003,19.0,346.071627
4,AFG,Afghanistan,2004,23.8,338.637274
